In [1]:
from src.data import Dataloader
import pandas as pd
import os
from src.utils import seed_everything
import random
from tqdm import tqdm
from typing import Callable
import torch.nn.functional as F
from src.data import TimeSeriesDataset
import torch
from src.nn import MLP, SlidingWindowBinaryClassification
from sklearn.metrics import roc_auc_score, precision_score


## Set up
To set up experience environment, we perform the following steps:

1. This section below will set up `const` needed for experiment
2. For reproducibility, we also set the same inital seed for everything (`numpy`, `torch`)

In [2]:
# Step 1.
DATA_PATH = os.path.abspath('data/clean/')
VAL_START_DATE = int(pd.Timestamp('2023-12-20').timestamp())  # Unix time in seconds
TEST_START_DATE = int(pd.Timestamp('2024-12-20').timestamp()) # Unix time in seconds
FEAT_COLUMN = ['Close','High','Low', 'Open']
BINARY_LABEL_COLUMN = 'price_increase'
REGRESSION_LABEL_COLUMN = 'next_close'

INIT_SEED = 720

# Step 2.
seed_everything(INIT_SEED)

## Prepare dataset for training

1. Shuffle all token networks (I.I.D)
2. Split datasets for training, validation and testing: the first 70% of datasets for training, the next 15% for validation, the next 15% for testing
3. Load all dataset in memory to `TimeSeriesDataset`

There are 49 tokens datasets in total, so we keep 33 networks for training, 16 tokens for validation and 16 tokens for testing.

In [3]:
files = os.listdir(DATA_PATH)
random.shuffle(files) # Step 1

# Step 2
train_token_list = files[:33]
valid_token_list = files[33:33+8]
test_token_list = files [-8:]

assert len(set(train_token_list).intersection(set(valid_token_list))) == 0
assert len(set(test_token_list).intersection(set(valid_token_list))) == 0

# Step 3
data_loader = Dataloader(DATA_PATH)

train_data = []
valid_data = []
test_data = []

for file_name in tqdm(train_token_list):
    train_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN))

for file_name in tqdm(valid_token_list):
    valid_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN))

for file_name in tqdm(test_token_list):
    test_data.append(data_loader.from_csv(file_name, feat_columns=FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN))

train_data_split = []
valid_data_split = []
test_data_split = []

for data in tqdm(train_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    train_data_split.append((train,val,test))

for data in tqdm(valid_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    valid_data_split.append((train,val,test))

for data in tqdm(test_data):
    train, val_test = data.split(VAL_START_DATE)
    val, test = val_test.split(TEST_START_DATE)
    test_data_split.append((train,val,test))

100%|██████████| 8/8 [00:00<00:00, 1073.12it/s]


In [5]:
def normalize(train_data: TimeSeriesDataset, valid_data: TimeSeriesDataset, test_data: TimeSeriesDataset, normalize_y = False):
    
    max_values,_ = torch.max(train_data.x, dim=0)
    min_values,_ = torch.min(train_data.x, dim=0)

    train_data.x = (train_data.x - min_values) / (max_values - min_values)
    valid_data.x = (valid_data.x - min_values) / (max_values - min_values)
    test_data.x = (test_data.x - min_values) / (max_values - min_values)

    if normalize_y:
        max_values,_ = torch.max(train_data.y, dim=0)
        min_values,_ = torch.min(train_data.y, dim=0)

        train_data.y = (train_data.y - min_values) / (max_values - min_values)
        valid_data.y = (valid_data.y - min_values) / (max_values - min_values)
        test_data.y = (test_data.y - min_values) / (max_values - min_values)

    return train_data,valid_data, test_data


data_loader = Dataloader(DATA_PATH)
timeseries_data = data_loader.from_csv("aave.csv", feat_columns = FEAT_COLUMN, label_column=BINARY_LABEL_COLUMN)

train_data, val_test_data = timeseries_data.split(VAL_START_DATE)
val_data, test_data = val_test_data.split(TEST_START_DATE)


train_data, val_data, test_data = normalize(train_data,val_data, test_data)
print(train_data.x.shape)



torch.Size([799, 4])


## Training loop

In [6]:



def train(train_data : TimeSeriesDataset, val_data : TimeSeriesDataset, model: Callable, criterion : Callable, opt: torch.optim.Optimizer, epoch = 50):

    for i in range(epoch):
        all_loss = []
        for time, feat, y in train_data:
            opt.zero_grad()
            z = model(feat.float())
            loss = criterion(
                z.float(),y.float()
            )
            all_loss.append(loss.item())
            loss.backward()
            opt.step()
        print(f"[INFO] Epoch - {i+1} : {sum(all_loss)/float(len(all_loss))}")


model = MLP(in_channel= 4, out_channel= 1, dim = 128, num_layers=3)

opt = torch.optim.Adam(
    model.parameters(), lr=float(0.001)
)


train(train_data, val_data, model, F.binary_cross_entropy_with_logits,opt)

           


[INFO] Epoch - 1 : 0.6956362474397366
[INFO] Epoch - 2 : 0.6945078531851309
[INFO] Epoch - 3 : 0.6936263258525815
[INFO] Epoch - 4 : 0.6932266503758961
[INFO] Epoch - 5 : 0.6930931296306796
[INFO] Epoch - 6 : 0.6926203023208695
[INFO] Epoch - 7 : 0.6926424404855663
[INFO] Epoch - 8 : 0.6921765046364375
[INFO] Epoch - 9 : 0.6918809149829258
[INFO] Epoch - 10 : 0.6918731558606382
[INFO] Epoch - 11 : 0.6915578293114043
[INFO] Epoch - 12 : 0.691320044152877
[INFO] Epoch - 13 : 0.6912001318791333
[INFO] Epoch - 14 : 0.6910913446743289
[INFO] Epoch - 15 : 0.6909290587573833
[INFO] Epoch - 16 : 0.6907807715022668
[INFO] Epoch - 17 : 0.6907525942456887
[INFO] Epoch - 18 : 0.6904824428698596
[INFO] Epoch - 19 : 0.6903988227751735
[INFO] Epoch - 20 : 0.690742892675913
[INFO] Epoch - 21 : 0.6904382647128219
[INFO] Epoch - 22 : 0.6902252480816632
[INFO] Epoch - 23 : 0.6900614763753435
[INFO] Epoch - 24 : 0.6900878008524974
[INFO] Epoch - 25 : 0.6897460895873727
[INFO] Epoch - 26 : 0.68989692074485

## Sliding window baseline

In [6]:
baseline = SlidingWindowBinaryClassification()

auc = []
for data in test_data_split:
    train, val, test = data
    preds = []
    for _, _, y in val:
        baseline.update(y.item())
    
    for _,_,y in test:
        y = baseline()
        preds.append(y)
        baseline.update(y) 
    print(test.y.shape)
    print(len(preds))
    score = roc_auc_score(test.y.squeeze().tolist(), preds)
    auc.append(score)

print(sum(auc)/len(auc))
    
    

torch.Size([381, 1])
381
torch.Size([381, 1])
381
torch.Size([381, 1])
381
torch.Size([381, 1])
381
torch.Size([381, 1])
381
torch.Size([381, 1])
381
torch.Size([381, 1])
381
torch.Size([381, 1])
381
0.5
